# Importing Libraries and Setup

In [22]:
import os

os.environ["HF_HOME"] = f"/rs1/researchers/a/amallav/models/hf_home"

os.environ["HF_HUB_CACHE"] = os.path.join(os.environ["HF_HOME"], "hub")
os.environ["HF_HUB_OFFLINE"] = "1"   # only after cache is populated

In [23]:
import json #Used later to store the top-k predictions as a JSON string in the final CSV
from pathlib import Path
import pandas as pd
import torch
from bioclip import TreeOfLifeClassifier, Rank #TreeOfLifeClassifier = the classifier that predicts biological taxa; Rank = tells the classifier what taxonomic level you want

In [24]:
# os.environ["PROJECT_DIR"] = "/rs1/researchers/a/amallav/"
# print(os.environ["PROJECT_DIR"])
# os.environ["OUTPUTS_DIR"] = "/rs1/researchers/a/amallav/results"
# print(os.environ["OUTPUTS_DIR"])
# os.environ["OUTPUTS_CP_DIR"] = "/rs1/researchers/a/amallav/results/outputs_crop"
# print(os.environ["OUTPUTS_CP_DIR"])
# os.environ["OUTPUTS_BCCP_DIR"] = "/rs1/researchers/a/amallav/results/outputs_bioclip_crop"
# print(os.environ["OUTPUTS_BCCP_DIR"])
# df = pd.read_csv("/rs1/researchers/a/amallav/results/outputs_crop/cropped_metadata.csv")
# # df["crop_file_path"][0]
# p = Path("/rs1/researchers/a/amallav/image_dataset/obs_341194952_photo_621071156.jpg")
# p.exists()

In [25]:
PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "/rs1/researchers/a/amallav/")).resolve()
OUTPUTS_DIR=Path(os.environ.get("OUTPUTS_DIR", "/rs1/researchers/a/amallav/results")).resolve()
IMAGE_DATASET_DIR = Path(os.getenv("IMAGE_DATASET_DIR", "/rs1/researchers/a/amallav/image_dataset")).resolve()

# OUTPUTS_CP_DIR = Path(os.environ.get("OUTPUTS_CP_DIR", OUTPUTS_DIR / "outputs_crop")).resolve()

OUTPUTS_BC_DIR = Path(os.environ.get("OUTPUTS_BC_DIR", OUTPUTS_DIR / "outputs_bioclip")).resolve()
OUTPUTS_BC_DIR.mkdir(parents=True, exist_ok=True)

# CROP_OUT = PROJECT_DIR / "outputs_crop"
META_CSV = OUTPUTS_DIR / "baseline_metadata.csv" #This is the metadata file that tells us what crop files exist.
OUT_CSV = OUTPUTS_BC_DIR / "bioclip_species_predictions.csv"
output_csv = OUTPUTS_BC_DIR / "predictions.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR, "| exists:", OUTPUTS_DIR.exists())
print("IMAGE_DATASET_DIR:", IMAGE_DATASET_DIR, "| exists:", IMAGE_DATASET_DIR.exists())
# print("OUTPUTS_CP_DIR:", OUTPUTS_CP_DIR, "| exists:", OUTPUTS_CP_DIR.exists())
print("OUTPUTS_BC_DIR:", OUTPUTS_BC_DIR, "| exists:", OUTPUTS_BC_DIR.exists())
print("META_CSV    :", META_CSV, "| exists:", META_CSV.exists())
print("OUT_CSV     :", OUT_CSV, "| exists:", OUT_CSV.exists())
print("output_csv     :", output_csv, "| exists:", output_csv.exists())

PROJECT_DIR: /rs1/researchers/a/amallav
OUTPUTS_DIR: /rs1/researchers/a/amallav/results | exists: True
IMAGE_DATASET_DIR: /rs1/researchers/a/amallav/image_dataset | exists: True
OUTPUTS_BC_DIR: /rs1/researchers/a/amallav/results/outputs_bioclip | exists: True
META_CSV    : /rs1/researchers/a/amallav/results/baseline_metadata.csv | exists: True
OUT_CSV     : /rs1/researchers/a/amallav/results/outputs_bioclip/bioclip_species_predictions.csv | exists: False
output_csv     : /rs1/researchers/a/amallav/results/outputs_bioclip/predictions.csv | exists: False


# Load Bioclip 

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [27]:
# model_dir = "/share/ftrscape/{}/models/bioclip".format(__import__("os").environ["USER"])

In [28]:
MODEL_STR = "hf-hub:imageomics/bioclip-2"

TOP_K = 5 #top 5 species predictions for each image
BATCH_SIZE = 1 #This controls how many cropped images are processed at once. Might have to tune this parameter

classifier = TreeOfLifeClassifier(
    device=device,
    model_str=MODEL_STR,
)

print("Device   :", device)
print("Model    :", MODEL_STR)
print("Top-k    :", TOP_K)
print("Batch size:", BATCH_SIZE)

Device   : cuda
Model    : hf-hub:imageomics/bioclip-2
Top-k    : 5
Batch size: 1


# Load CSV

In [36]:
assert META_CSV.exists(), f"metadata.csv not found: {META_CSV}"

meta = pd.read_csv(META_CSV)
# meta
required_cols = {"image_path", "image_id"}
missing_cols = required_cols - set(meta.columns)
assert not missing_cols, f"metadata.csv is missing columns: {missing_cols}"

In [35]:
# meta

,image_id,image_path
0,obs_341194952_photo_621071156,/rs1/researchers/a/amallav/image_dataset/obs_3...
1,obs_341194952_photo_621071168,/rs1/researchers/a/amallav/image_dataset/obs_3...
2,obs_341194952_photo_621071177,/rs1/researchers/a/amallav/image_dataset/obs_3...
3,obs_341194952_photo_621071190,/rs1/researchers/a/amallav/image_dataset/obs_3...
4,obs_341194952_photo_621071213,/rs1/researchers/a/amallav/image_dataset/obs_3...
...,...,...
24802,obs_414645_photo_28352912,/rs1/researchers/a/amallav/image_dataset/obs_4...
24803,obs_338604_photo_421440,/rs1/researchers/a/amallav/image_dataset/obs_3...
24804,obs_338604_photo_421441,/rs1/researchers/a/amallav/image_dataset/obs_3...
24805,obs_338604_photo_421442,/rs1/researchers/a/amallav/image_dataset/obs_3...


In [37]:
# #converts paths into absolute paths.
# def resolve_path(p):
#     if pd.isna(p):
#         return None
#     p = str(p).strip()
#     if os.path.isabs(p):
#         return p
#     return str((PROJECT_DIR / p).resolve())

# meta["masked_crop_path"] = meta["masked_crop_path"].apply(resolve_path)

In [38]:
meta = meta[meta["image_path"].notna()].copy() #remove rows where crop path is missing
meta["crop_exists"] = meta["image_path"].apply(os.path.exists)

missing_count = (~meta["crop_exists"]).sum()
if missing_count > 0:
    print(f"Skipping {missing_count} rows because crop file does not exist.")

meta = meta[meta["crop_exists"]].copy()  #only rows where the crop file actually exists.



In [39]:
keep_cols = ["image_path", "image_id"]
if "box_num" in meta.columns:
    keep_cols.append("box_num")

org_df = meta[keep_cols].drop_duplicates(subset=["image_path"]).reset_index(drop=True) #creates a clean crop table without deduplicates

print("Total valid images:", len(org_df))
org_df.head()

Total valid images: 24807


,image_path,image_id
0,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156
1,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071168
2,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071177
3,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071190
4,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071213


# Run species classification on all cropped images

In [40]:
# test = classifier.predict(
#     images="/gpfs_common/share03/ftrscape/snair3/wew_notebooks/outputs_crop/obs_250956763_photo_448935283_box2_copy.JPG",
#     rank=Rank.SPECIES, #Predict at species level.
#     k=TOP_K, #Return the top 5 predictions for each image.
#     batch_size=BATCH_SIZE, #Process 1 image at a time.
# )
# test_df=pd.DataFrame(test)

# # print("Total prediction rows:", len(pred_df))
# test_df.head()

In [41]:
# crop_paths = crop_df["crop_file_path"].tolist() #Collect all crop image paths into a list.

# predictions = classifier.predict(
#     images=crop_paths, #Pass the cropped image files to the model.
#     rank=Rank.SPECIES, #Predict at species level.
#     k=TOP_K, #Return the top 5 predictions for each image.
#     batch_size=BATCH_SIZE, #Process 1 image at a time.
# )

# pred_df = pd.DataFrame(predictions) #Turns the prediction output into a table.

# print("Total prediction rows:", len(pred_df))
# pred_df.head()

In [ ]:



# Create CSV with header only once

if not os.path.exists(output_csv):
    empty_df = pd.DataFrame(columns=[
        "file_name",
        "kingdom",
        "phylum",
        "class",
        "order",
        "family",
        "genus",
        "species_epithet",
        "species",
        "common_name",
        "score",
        "path",
        "id"
    ])
    empty_df.to_csv(output_csv, index=False)


for idx, row in org_df.iterrows():
    image_path = row["image_path"]

    # Run prediction for ONE image
    prediction = classifier.predict(
        images=[image_path],           # single image
        rank=Rank.SPECIES,
        k=TOP_K,
        batch_size=1
    )

    # Convert to DataFrame
    pred_df = pd.DataFrame(prediction)

    # (Optional) keep track of source image
    pred_df["image_path"] = image_path
    pred_df["image_index"] = idx

    # Append to CSV (no header after first time)
    pred_df.to_csv(
        output_csv,
        mode="a",
        header=not os.path.getsize(output_csv),
        index=False
    )

    print(f"✅ Processed {image_path} - {idx + 1}/{len(org_df)}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.59images/s]


✅ Processed /rs1/researchers/a/amallav/image_dataset/obs_341194952_photo_621071156.jpg - 1/24807


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.04images/s]


✅ Processed /rs1/researchers/a/amallav/image_dataset/obs_341194952_photo_621071168.jpg - 2/24807


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.80images/s]


...

✅ Processed /rs1/researchers/a/amallav/image_dataset/obs_337632558_photo_613819538.jpg - 403/24807


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.38images/s]


✅ Processed /rs1/researchers/a/amallav/image_dataset/obs_337632558_photo_613819539.jpg - 404/24807


  0%|                                                                                                                                         | 0/1 [00:00<?, ?images/s]


KeyboardInterrupt: 

In [44]:
pred_df = pd.read_csv(output_csv)
pred_df

,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score,path,id
0,/rs1/researchers/a/amallav/image_dataset/obs_3...,Animalia,Chordata,Mammalia,Rodentia,Geomyidae,Thomomys,Plesiothomomys potomacensis,Thomomys Plesiothomomys potomacensis,NaN,0.444246,/rs1/researchers/a/amallav/image_dataset/obs_3...,0
1,/rs1/researchers/a/amallav/image_dataset/obs_3...,Animalia,Chordata,NaN,Perciformes,Gobiidae,Gillichthys,mirabilis,Gillichthys mirabilis,Longjaw mudsucker,0.089155,/rs1/researchers/a/amallav/image_dataset/obs_3...,0
2,/rs1/researchers/a/amallav/image_dataset/obs_3...,Animalia,Chordata,Mammalia,Rodentia,Geomyidae,Thomomys,bottae,Thomomys bottae,Botta's pocket gopher,0.077021,/rs1/researchers/a/amallav/image_dataset/obs_3...,0
3,/rs1/researchers/a/amallav/image_dataset/obs_3...,Animalia,Chordata,NaN,Siluriformes,Clariidae,Clarias,batrachus,Clarias batrachus,Walking catfish,0.024569,/rs1/researchers/a/amallav/image_dataset/obs_3...,0
4,/rs1/researchers/a/amallav/image_dataset/obs_3...,Animalia,Chordata,Amphibia,Caudata,Ambystomatidae,Ambystoma,californiense,Ambystoma californiense,California tiger salamander,0.020836,/rs1/researchers/a/amallav/image_dataset/obs_3...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020,/rs1/researchers/a/amallav/image_dataset/obs_3...,Plantae,Tracheophyta,Magnoliopsida,Asterales,Asteraceae,Silphium,perfoliatum,Silphium perfoliatum,Cup Plant,0.408247,/rs1/researchers/a/amallav/image_dataset/obs_3...,403
2021,/rs1/researchers/a/amallav/image_dataset/obs_3...,Animalia,Arthropoda,Insecta,Lepidoptera,Papilionidae,Papilio,glaucus,Papilio glaucus,Tiger swallowtail,0.240340,/rs1/researchers/a/amallav/image_dataset/obs_3...,403
2022,/rs1/researchers/a/amallav/image_dataset/obs_3...,Plantae,Tracheophyta,Magnoliopsida,Asterales,Asteraceae,Silphium,perplexum,Silphium perplexum,NaN,0.084591,/rs1/researchers/a/amallav/image_dataset/obs_3...,403
2023,/rs1/researchers/a/amallav/image_dataset/obs_3...,Plantae,Tracheophyta,Magnoliopsida,Asterales,Asteraceae,Silphium,integrifolium,Silphium integrifolium,Whole-leaf rosinweed,0.057766,/rs1/researchers/a/amallav/image_dataset/obs_3...,403


# Convert top-k predictions into one row per cropped image

In [45]:
assert "file_name" in pred_df.columns, "Expected 'file_name' in prediction output"
assert "score" in pred_df.columns, "Expected 'score' in prediction output"
assert "species" in pred_df.columns, "Expected 'species' in prediction output"

In [46]:
# pred_df

In [ ]:
summary_rows = []

for image_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    row = {
        "image_path": image_path,
        "top1_species": top1.get("species"),
        "top2_species": top2.get("species"),
        "top3_species": top3.get("species"),
        "top4_species": top4.get("species"),
        "top5_species": top5.get("species"),
        "top1_common_name": top1.get("common_name"),
        "top1_score": top1.get("score"),
        "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

final_df = org_df.merge(summary_df, on="image_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(final_df))
final_df.head()

Final rows: 24807


,image_path,image_id,top1_species,top2_species,top3_species,top4_species,top5_species,top1_common_name,top1_score,topk_predictions_json
0,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156,Thomomys Plesiothomomys potomacensis,Thomomys Plesiothomomys potomacensis,Gillichthys mirabilis,Gillichthys mirabilis,Thomomys bottae,NaN,0.444246,"[{""species"": ""Thomomys Plesiothomomys potomace..."
1,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071168,Thomomys Plesiothomomys potomacensis,Thomomys bottae,Ardea alba,Hypsopsetta guttulata,Ardea modesta,NaN,0.201480,"[{""species"": ""Thomomys Plesiothomomys potomace..."
2,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071177,Ardea alba,Ardea modesta,Egretta thula,Egretta intermedia,Bubulcus ibis,Great Egret,0.674656,"[{""species"": ""Ardea alba"", ""common_name"": ""Gre..."
3,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071190,Ardea modesta,Ardea alba,Egretta intermedia,Bubulcus ibis,Egretta thula,Great Egret,0.666560,"[{""species"": ""Ardea modesta"", ""common_name"": ""..."
4,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071213,Ardea modesta,Ardea alba,Egretta intermedia,Egretta thula,Bubulcus ibis,Great Egret,0.618911,"[{""species"": ""Ardea modesta"", ""common_name"": ""..."


In [52]:
# final_df[final_df["top1_species"].isna()]

,image_path,image_id,top1_species,top2_species,top3_species,top4_species,top5_species,top1_common_name,top1_score,topk_predictions_json
404,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_337632555_photo_613819514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_337632541_photo_613819362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_337632541_photo_613819364,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
407,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_337632541_photo_613819379,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
408,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_337613554_photo_613782306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
24802,/rs1/researchers/a/amallav/image_dataset/obs_4...,obs_414645_photo_28352912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24803,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_338604_photo_421440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24804,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_338604_photo_421441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24805,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_338604_photo_421442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
summary_rows = []

for image_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    for i in range(5):
        no=i+1
        no=str(no)
        val="top"+no+"_species"
        sc="top"+no+"_score"
        cc="top"+no+"_common_name"
        row = {
            "image_path": image_path,
            # val:group.iloc[i].get("species"),
            "species": group.iloc[i].get("species"),
            # "top2_species": top2.get("species"),
            # "top3_species": top3.get("species"),
            # "top4_species": top4.get("species"),
            # "top5_species": top5.get("species"),
            "common_name": group.iloc[i].get("common_name"),
            "kingdom": group.iloc[i].get("kingdom"),
            "phylum": group.iloc[i].get("phylum"),
            "class": group.iloc[i].get("class"),
            "order": group.iloc[i].get("order"),
            "family": group.iloc[i].get("family"),
            "genus": group.iloc[i].get("genus"),
            "score": group.iloc[i].get("score"),
            "top_no":no,
            "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

full_df = org_df.merge(summary_df, on="image_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(full_df))
full_df.head()

Final rows: 26423


,image_path,image_id,species,common_name,kingdom,phylum,class,order,family,genus,score,top_no,topk_predictions_json
0,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156,Thomomys Plesiothomomys potomacensis,NaN,Animalia,Chordata,Mammalia,Rodentia,Geomyidae,Thomomys,0.444246,1,"[{""species"": ""Thomomys Plesiothomomys potomace..."
1,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156,Thomomys Plesiothomomys potomacensis,NaN,Animalia,Chordata,Mammalia,Rodentia,Geomyidae,Thomomys,0.444246,2,"[{""species"": ""Thomomys Plesiothomomys potomace..."
2,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156,Gillichthys mirabilis,Longjaw mudsucker,Animalia,Chordata,NaN,Perciformes,Gobiidae,Gillichthys,0.089155,3,"[{""species"": ""Thomomys Plesiothomomys potomace..."
3,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156,Gillichthys mirabilis,Longjaw mudsucker,Animalia,Chordata,NaN,Perciformes,Gobiidae,Gillichthys,0.089155,4,"[{""species"": ""Thomomys Plesiothomomys potomace..."
4,/rs1/researchers/a/amallav/image_dataset/obs_3...,obs_341194952_photo_621071156,Thomomys bottae,Botta's pocket gopher,Animalia,Chordata,Mammalia,Rodentia,Geomyidae,Thomomys,0.077021,5,"[{""species"": ""Thomomys Plesiothomomys potomace..."


# Save final output CSV

In [ ]:
OUT_CSV = OUTPUTS_BC_DIR / "bioclip_species_predictions_baseline.csv"
final_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /rs1/researchers/a/amallav/results/outputs_bioclip_crop/bioclip_species_predictions_2.csv


In [ ]:
OUT_CSV = OUTPUTS_BC_DIR / "results_baseline.csv"
full_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /rs1/researchers/a/amallav/results/outputs_bioclip_crop/results_crop_2.csv
